# Warm-up: Generation Parameters and Prompting — Solutions

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

A set of short, easy exercises that follow the *Basics of Text Generation* and *Prompting LLMs* demos. Each exercise gives you a **goal** and a piece of code that does **not** reach it yet — usually because a generation parameter is missing, too high or too low, or because the prompt is not clear enough.

Your job: change the code marked with `# <-- FIX` or `# TODO`, run the cell, and read the output. A small `check()` at the end of each cell tells you whether you got there.

> The checks are simple heuristics. A language model is not perfectly predictable, so if the output clearly meets the goal but the check disagrees, trust your own reading.

**Part A — Generation parameters**
1. `max_new_tokens` too low
2. Length belongs in the prompt, not in `max_new_tokens`
3. Reproducible labels: `do_sample`
4. `temperature` too high
5. `temperature` too low
6. `top_k` too low
7. Find the hidden culprit (`top_p`)
8. `repetition_penalty` too high
9. `set_seed` for reproducible sampling
10. Pick the parameters yourself

**Part B — Prompting**

11. Zero-shot: answer with the label only
12. System prompt: control the format
13. Few-shot: fill in the examples
14. Chain-of-thought needs room to think
15. Structured output: valid JSON

## Setup

The same model and helpers as in the demos. On a free Colab / Kaggle GPU loading takes about a minute.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import json
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

from transformers import pipeline, set_seed
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

pipe = pipeline(
    task="text-generation",
    model=MODEL_NAME,
    dtype="auto",
    device_map="auto",
)
print(f"{MODEL_NAME} on {pipe.model.device}")

In [ ]:
def generate(prompt, system="You are a helpful assistant.", **kwargs):
    """One completion. Extra keyword arguments are forwarded to generate()."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]
    result = pipe(messages, **kwargs)
    return result[0]["generated_text"][-1]["content"]


def generate_n(prompt, n, system="You are a helpful assistant.", **kwargs):
    """n completions of the same prompt, returned as a list of strings."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]
    result = pipe(messages, num_return_sequences=n, **kwargs)
    return [r["generated_text"][-1]["content"] for r in result]


def check(ok, hint):
    """Print a tick if the goal is met, otherwise a hint."""
    print("\n✅ Goal reached!" if ok else f"\n❌ Not yet. Hint: {hint}")


def ends_cleanly(text):
    return text.strip().endswith((".", "!", "?"))

---
# Part A — Generation Parameters

Quick reminder:

| Parameter | What it does | Typical values |
|---|---|---|
| `max_new_tokens` | Hard cap on the number of generated tokens | 20 – 500 |
| `do_sample` | `False` = greedy (always the most likely token), `True` = sample | — |
| `temperature` | < 1 sharper / safer, > 1 flatter / wilder | 0.3 – 1.2 |
| `top_k` | Sample only from the *k* most likely tokens | 20 – 100 |
| `top_p` | Sample only from the smallest set with cumulative probability ≥ *p* | 0.8 – 0.95 |
| `repetition_penalty` | Push down tokens that already appeared (1.0 = off) | 1.0 – 1.2 |

## Exercise 1 — The answer is cut off

**Goal:** a complete two-sentence answer.

In [ ]:
PROMPT = "Explain in two sentences what a large language model is."

params = dict(do_sample=False, max_new_tokens=120)   # 8 tokens is only a few words

answer = generate(PROMPT, **params)
print(answer)

check(ends_cleanly(answer),
      "the answer stops mid-sentence. Which parameter limits the output length?")

## Exercise 2 — Short, but not truncated

We want a **short** answer (at most 30 words). Someone tried to get it by lowering `max_new_tokens` — and got a sentence that just stops.

**Goal:** a complete answer of at most 30 words. Keep `max_new_tokens` at 30 or more — shorten the answer through the **prompt** instead.

In [ ]:
PROMPT = "Explain in one short sentence what a large language model is."

params = dict(do_sample=False, max_new_tokens=60)

answer = generate(PROMPT, **params)
print(answer)
print(f"\n({len(answer.split())} words)")

check(ends_cleanly(answer) and len(answer.split()) <= 30 and params["max_new_tokens"] >= 30,
      "max_new_tokens only cuts the text off. Ask for the length you want in the prompt, "
      "and give the model enough tokens to finish.")

## Exercise 3 — The same comment gets different labels

We classify a survey answer five times. For a coding task the label must **not** change between runs.

**Goal:** all five runs return the same label.

In [ ]:
PROMPT = ("Classify this survey answer as positive, neutral or negative. "
          "Answer with the label only.\n\n"
          "Answer: 'The bus is sometimes late, but overall it gets me to work.'")

params = dict(do_sample=False, max_new_tokens=5)   # greedy decoding is deterministic

labels = [generate(PROMPT, **params).strip() for _ in range(5)]
print(labels)

check(len(set(labels)) == 1 and not params["do_sample"],
      "the model is sampling. How do you make it always pick the most likely token?")

## Exercise 4 — The story is gibberish

**Goal:** a creative but readable story opening. Keep sampling on (`do_sample=True`).

In [ ]:
PROMPT = "Write the first three sentences of a story about a lighthouse keeper who finds a message in a bottle."

params = dict(do_sample=True, temperature=0.8, max_new_tokens=100)

set_seed(0)
print(generate(PROMPT, **params))

check(params["do_sample"] and 0.5 <= params["temperature"] <= 1.2,
      "a very high temperature flattens the distribution so that nonsense tokens get picked.")

## Exercise 5 — Brainstorming returns five copies

We ask for five survey titles and want five **different** ideas.

**Goal:** at least 4 distinct titles out of 5.

In [ ]:
PROMPT = "Suggest a title for a survey about how students use public transport. Title only."

params = dict(do_sample=True, temperature=1.0, max_new_tokens=25)

set_seed(0)
titles = generate_n(PROMPT, n=5, **params)
for t in titles:
    print("-", t)

check(len(set(titles)) >= 4,
      "with a temperature this close to 0 sampling behaves like greedy decoding.")

## Exercise 6 — Sampling, but still no variety

Temperature is fine this time, yet all titles are the same.

**Goal:** at least 4 distinct titles out of 5.

In [ ]:
PROMPT = "Suggest a title for a survey about how students use public transport. Title only."

params = dict(do_sample=True, temperature=1.0, top_k=50, max_new_tokens=25)   # top_k=1 is greedy

set_seed(0)
titles = generate_n(PROMPT, n=5, **params)
for t in titles:
    print("-", t)

check(len(set(titles)) >= 4,
      "how many candidate tokens does top_k=1 leave to choose from?")

## Exercise 7 — Find the hidden culprit

Same goal again. `temperature` and `top_k` look reasonable now — one other value is still killing the variety. Find it and fix it.

**Goal:** at least 4 distinct titles out of 5.

In [ ]:
PROMPT = "Suggest a title for a survey about how students use public transport. Title only."

params = dict(do_sample=True, temperature=1.0, top_k=50, top_p=0.95, max_new_tokens=25)

set_seed(0)
titles = generate_n(PROMPT, n=5, **params)
for t in titles:
    print("-", t)

check(len(set(titles)) >= 4,
      "top_p keeps the smallest set of tokens whose probabilities add up to p. "
      "How many tokens are needed to reach 1 %?")

With `top_p=0.01` only the single most likely token survives at every step — just like `top_k=1`. Several parameters can each switch off diversity on their own, so when outputs look suspiciously identical, check all of them.

## Exercise 8 — Strange vocabulary

Someone wanted to avoid repeated words and turned `repetition_penalty` up a lot. Now the model avoids even normal words like *the* and *and*, and the text goes strange.

**Goal:** natural text with only a mild penalty.

In [ ]:
PROMPT = "Describe a busy city market in the morning in about four sentences."

params = dict(do_sample=True, temperature=0.7, repetition_penalty=1.1, max_new_tokens=120)

set_seed(0)
print(generate(PROMPT, **params))

check(1.0 <= params["repetition_penalty"] <= 1.3,
      "1.0 means no penalty; values above ~1.3 start to punish ordinary function words.")

## Exercise 9 — Reproducible sampling

Sometimes you *want* sampling (e.g. for creative text) but still need to reproduce a result later, for a report or a colleague.

**Goal:** keep `do_sample=True`, but make both runs return exactly the same text. Insert one line before each call.

In [ ]:
PROMPT = "Write a one-sentence slogan for a city bike-sharing scheme."
params = dict(do_sample=True, temperature=1.0, max_new_tokens=30)

set_seed(42)
run_1 = generate(PROMPT, **params)

set_seed(42)
run_2 = generate(PROMPT, **params)

print("RUN 1:", run_1)
print("RUN 2:", run_2)

check(params["do_sample"] and run_1 == run_2,
      "fix the random number generator with set_seed(...) — same seed before both calls.")

## Exercise 10 — Pick the parameters yourself

Three tasks, three empty parameter sets. Fill in the `None` values.

| Task | What we want |
|---|---|
| (a) Extract the date from a sentence | always the same, correct answer |
| (b) Write a short poem | creative, varied, but readable |
| (c) Draft survey questions | **3** different drafts from one call |

In [ ]:
# (a) extraction: greedy, deterministic
params_a = dict(do_sample=False, max_new_tokens=15)
print("(a)", generate("Extract the date from this sentence, answer with the date only: "
                      "'The interview took place on 14 March 2024 in Ljubljana.'", **params_a))

# (b) poem: sampling with a moderate temperature
params_b = dict(do_sample=True, temperature=0.8, max_new_tokens=80)
set_seed(0)
print("\n(b)", generate("Write a four-line poem about autumn.", **params_b))

# (c) three drafts from one call
params_c = dict(do_sample=True, temperature=0.9, max_new_tokens=40)
drafts = generate_n("Write one survey question about remote work satisfaction.",
                    n=3, **params_c)
print("\n(c)")
for d in drafts:
    print("-", d)

check(params_a["do_sample"] is False
      and params_b["temperature"] is not None and 0.5 <= params_b["temperature"] <= 1.2
      and len(drafts) == 3,
      "(a) extraction should be greedy, (b) needs a moderate temperature, (c) needs n=3.")

---
# Part B — Prompting

From here on decoding is greedy (`do_sample=False`), so every change you see comes from the **prompt**. Edit the prompt (and, where marked, `max_new_tokens`).

## Exercise 11 — Just the label, please

**Goal:** the model replies with exactly one word: `positive`, `neutral` or `negative`.

In [ ]:
LABELS = {"positive", "neutral", "negative"}

REVIEW = "The room was dirty and nobody answered the phone at reception."

PROMPT = (f"Classify the sentiment of this hotel review as positive, neutral or negative. "
          f"Answer with the label only.\n\nReview: {REVIEW}")

answer = generate(PROMPT, do_sample=False, max_new_tokens=60)
print(repr(answer))

check(answer.strip().strip(".").lower() in LABELS,
      "name the three allowed labels in the prompt and say 'Answer with the label only.'")

## Exercise 12 — Control the format with a system prompt

Change **only** the `SYSTEM` message, not the user question.

**Goal:** the answer is exactly 3 bullet points, each starting with `- `, and nothing else.

In [ ]:
SYSTEM = ("You are a concise assistant. Always answer with exactly 3 bullet points. "
          "Each bullet starts with '- ' and is one short sentence. "
          "Write nothing before or after the bullets.")

answer = generate("Give me tips for writing a good survey question.",
                  system=SYSTEM, do_sample=False, max_new_tokens=200)
print(answer)

lines = [l for l in answer.strip().splitlines() if l.strip()]
check(len(lines) == 3 and all(l.startswith("- ") for l in lines),
      "tell the model in the system prompt exactly what shape every answer must have.")

## Exercise 13 — Few-shot: fill in the examples

We sort open survey answers into three topics: `transport`, `housing`, `health`. The few-shot prompt is ready, but the example labels are missing.

**Goal:** fill in the three `TODO`s so the model answers the last item with one of the three topics.

In [ ]:
TOPICS = {"transport", "housing", "health"}

FEW_SHOT = """Assign each survey answer to one topic: transport, housing or health.

Answer: 'Rents in the centre have doubled in five years.'
Topic: housing

Answer: 'The waiting list for a family doctor is far too long.'
Topic: health

Answer: 'There is no night bus from the university to my neighbourhood.'
Topic: transport

Answer: 'I cycle to work but the bike lanes end in the middle of nowhere.'
Topic:"""

answer = generate(FEW_SHOT, do_sample=False, max_new_tokens=10)
print(repr(answer))

check("TODO" not in FEW_SHOT and answer.strip().strip(".").lower() in TOPICS,
      "replace each TODO with the correct topic; the model copies the pattern it sees.")

## Exercise 14 — Let it think (and give it room)

Two things are wrong here: the prompt does not invite step-by-step reasoning, **and** the token budget leaves no room for it.

**Goal:** the answer contains the correct change, **1.40 €**.

In [ ]:
PROBLEM = ("A café sells coffee for 1.20 € a cup. Ana buys 3 cups and pays with a 5 € note. "
           "How much change does she get?")

PROMPT = PROBLEM + "\nLet's think step by step, then give the final answer."

answer = generate(PROMPT, do_sample=False, max_new_tokens=300)
print(answer)

check("1.4" in answer or "1,4" in answer,
      "ask the model to think step by step, and raise max_new_tokens so the reasoning fits.")

## Exercise 15 — Valid JSON with the right keys

We want to turn a sentence into a data row. The helper `parse_json()` pulls out the part between the first `{` and the last `}` and parses it.

**Goal:** the reply parses as JSON and has exactly the keys `name`, `age` and `city`.

In [ ]:
def parse_json(text):
    try:
        return json.loads(text[text.index("{"): text.rindex("}") + 1])
    except ValueError:
        return None


SENTENCE = "Maja Novak is a 34-year-old nurse who lives in Maribor."

PROMPT = ("Extract information about the person. Return JSON only, with exactly these keys: "
          '"name" (string), "age" (integer), "city" (string).\n\n'
          f"{SENTENCE}")

answer = generate(PROMPT, do_sample=False, max_new_tokens=60)
print(answer)

row = parse_json(answer)
print("\nParsed:", row)

check(isinstance(row, dict) and set(row) == {"name", "age", "city"},
      "(1) name the exact keys you want in the prompt, "
      "(2) 10 tokens is not enough for a whole JSON object.")

---
## Done!

What to take away:

* **Length:** `max_new_tokens` is a safety cap, not a length instruction — ask for the length in the prompt.
* **Deterministic tasks** (labels, extraction): `do_sample=False`.
* **Creative tasks:** `do_sample=True` with a moderate `temperature` (≈0.7–1.0), `top_k`≈50, `top_p`≈0.9–0.95.
* Extreme values of `temperature`, `top_k`, `top_p` or `repetition_penalty` either kill diversity or produce nonsense.
* `set_seed()` makes sampled output reproducible.
* **Prompting:** state the allowed answers, the format and the keys explicitly; use a system prompt for rules, examples for patterns, and "think step by step" (with enough tokens) for reasoning.